# Convolutional Neural Network — Cat vs Dog Classifier

## Why Standard Neural Networks Fail at Images

A 64x64 colour image has 64 x 64 x 3 = **12,288 pixels**. If we flattened it into a vector and fed it to a fully-connected layer with 128 neurons, that single layer would have 12,288 x 128 = **1.57 million parameters**.

Problems with this approach:
- **No spatial awareness:** The network treats pixel (0,0) and pixel (63,63) as completely independent features. It cannot learn that nearby pixels form shapes.
- **Parameter explosion:** 1.57M parameters from one layer on a 64x64 image. Real images are 224x224 or larger.
- **No translation invariance:** A cat in the top-left corner looks completely different from a cat in the bottom-right corner as raw pixel vectors.

---

## What CNNs Do Differently

A Convolutional Neural Network processes images with **local, shared filters**:

1. **Convolution layers** slide small filters (e.g., 3x3 pixels) across the image, detecting local patterns like edges, corners, and textures
2. **Pooling layers** reduce spatial size, making the network tolerant to small position shifts
3. **Multiple layers** build hierarchically: edges → shapes → object parts → full objects

A 3x3 filter applied to an image of any size has only **9 weights** — shared across all positions. This is why CNNs can handle high-resolution images efficiently.

---

## The Architecture We Will Build

```
Input: 64x64x3 image
         ↓
Conv2D (32 filters, 3x3, ReLU)    → detect edges and textures
         ↓
MaxPooling (2x2)                   → reduce spatial size, retain dominant features
         ↓
Conv2D (32 filters, 3x3, ReLU)    → detect higher-level patterns
         ↓
MaxPooling (2x2)
         ↓
Flatten                            → convert feature maps to 1D vector
         ↓
Dense (128 neurons, ReLU)          → combine all detected features
         ↓
Dense (1 neuron, Sigmoid)          → P(dog) — binary classification output
```

---

## The Task

Binary image classification: given a photo, predict whether it contains a **cat** or a **dog**.
- Training set: 8,000 images (4,000 cats, 4,000 dogs)
- Test set: 2,000 images (1,000 cats, 1,000 dogs)

### Import Libraries

| Library | Why we need it |
|---------|---------------|
| `tensorflow` | Building and training the CNN |
| `ImageDataGenerator` | Loading images in batches with augmentation — handles datasets too large to fit in memory |

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
tf.__version__

## Part 1: Data Preprocessing for Images

Image data requires different preprocessing than tabular data. Instead of `pd.read_csv()`, we use `ImageDataGenerator` — a Keras utility that:

1. Loads images from disk in batches (avoids memory overflow)
2. Automatically labels images based on the subdirectory they are in (`training_set/cats/` → label 0, `training_set/dogs/` → label 1)
3. Applies augmentation to training images to artificially expand the dataset

### Preprocessing the Training Set — With Data Augmentation

**`rescale=1./255`** — Normalises pixel values from the range [0, 255] to [0, 1]. Neural networks train better with small input values (same reason we use StandardScaler for tabular data).

**Data Augmentation — why it is essential:**

With 8,000 training images, a CNN can easily memorise the training set (overfit). Augmentation artificially creates new training examples by randomly transforming existing ones:

| Augmentation | What it does | Why it helps |
|-------------|--------------|-------------|
| `shear_range=0.2` | Shears the image diagonally | Teaches invariance to slight geometric distortions |
| `zoom_range=0.2` | Randomly zooms in/out | Teaches scale invariance |
| `horizontal_flip=True` | Randomly mirrors the image | A cat facing left = a cat facing right |

Each training epoch sees a slightly different version of every image. The model cannot memorise — it must generalise.

**`target_size=(64, 64)`** — Resizes all images to 64x64 pixels for uniform input shape.

**`batch_size=32`** — Loads 32 images at a time during training.

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)
training_set = train_datagen.flow_from_directory('dataset/training_set',
                                                 target_size = (64, 64),
                                                 batch_size = 32,
                                                 class_mode = 'binary')

### Preprocessing the Test Set — No Augmentation

The test set uses **only rescaling** — no augmentation.

**Why no augmentation on the test set?**

Augmentation is a regularisation technique to improve training generalisation. The test set simulates real-world deployment — we evaluate the model on natural, un-augmented images to get an honest performance estimate.

Augmenting test images would give different accuracy depending on the random transformations applied — a noisy, unreliable metric.

In [ ]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('dataset/test_set',
                                            target_size = (64, 64),
                                            batch_size = 32,
                                            class_mode = 'binary')

## Part 2: Building the CNN Architecture

We build the network layer by layer using `Sequential`. Each layer is added in the order data flows through the network during forward propagation.

### Initialise the Network

`Sequential()` creates an empty container for layers. We will add them one by one in the order they execute.

In [ ]:
cnn = tf.keras.models.Sequential()

### Layer 1: Convolution

```python
Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3])
```

**What convolution does:**
A 3x3 filter slides across the image one position at a time. At each position, it computes the dot product between the filter weights and the 3x3 patch of pixels beneath it. The result is a single number — how strongly that pattern was detected at that location.

After sliding across the entire 64x64 image, the filter produces a 62x62 **feature map** (slightly smaller due to edge handling).

**`filters=32`** — 32 different 3x3 filters, each learning to detect a different low-level pattern: horizontal edges, vertical edges, diagonals, colour transitions, etc. This produces 32 feature maps.

**`activation='relu'`** — Applied element-wise to the feature map. Negative values (pattern not detected) become 0; positive values (pattern detected) pass through unchanged.

**`input_shape=[64, 64, 3]`** — Required only for the first layer: 64x64 pixels, 3 colour channels (RGB).

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

### Layer 2: Max Pooling

```python
MaxPool2D(pool_size=2, strides=2)
```

**What max pooling does:**
Divides each feature map into 2x2 non-overlapping windows and keeps only the maximum value in each window.

This halves the spatial dimensions: a 62x62 feature map becomes 31x31.

**Why max pooling?**

1. **Reduces computation:** Each subsequent layer processes a smaller spatial grid
2. **Translation invariance:** A pattern detected at position (10,10) vs (11,10) both produce a high value in the same max pool window — small spatial shifts do not change the output
3. **Mild overfitting reduction:** Reduces the number of parameters the network must learn

The maximum value is kept because we care about whether a pattern *was* detected (max activation), not its exact location.

In [ ]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Second Convolutional + Pooling Block

Adding a second convolutional layer allows the network to learn **higher-order features**:

- **Layer 1 convolution** detected: edges, corners, colour patches
- **Layer 2 convolution** combines those: curves, textures, object parts (ear shape, fur pattern)

Deeper networks learn progressively more abstract representations. Real-world CNNs for image classification (VGG, ResNet) have 16-152 layers.

The second `MaxPool2D` further reduces spatial size: 31x31 → 15x15 (with integer rounding).

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3: Flatten

After two convolutional + pooling blocks, we have 32 feature maps of size approximately 15x15 = **7,200 numbers**.

`Flatten()` converts this 3D tensor (height x width x filters) into a 1D vector of 7,200 numbers that can be fed into a standard Dense layer.

Think of it as: we have detected all the relevant local patterns in the image — now we need to combine them to make a global prediction.

In [ ]:
cnn.add(tf.keras.layers.Flatten())

### Step 4: Fully Connected Layer

```python
Dense(units=128, activation='relu')
```

This is identical to the hidden layers in the ANN notebook. Every neuron connects to all 7,200 flattened features.

The Dense layer's job: learn which **combinations** of detected patterns indicate cat vs dog. For example:
- Pointy ears + whisker texture + certain eye shape → cat
- Floppy ears + snout shape + different fur texture → dog

128 neurons provide enough capacity to learn complex combinations without excessive overfitting.

In [ ]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

### Step 5: Output Layer

```python
Dense(units=1, activation='sigmoid')
```

Binary classification — one output neuron with sigmoid activation outputs P(dog).

- Output > 0.5 → predicted dog
- Output ≤ 0.5 → predicted cat

This is the same output design as the ANN notebook.

In [ ]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3: Training the CNN

Training a CNN on images follows the same forward pass → loss computation → backpropagation loop as the ANN.

The key difference: gradients flow backward through the Dense layers *and* through the convolutional filters. The filters update their weights to become better pattern detectors.

### Compile the Network

Same compilation as the ANN:
- **`optimizer='adam'`** — adaptive learning rate optimiser
- **`loss='binary_crossentropy'`** — correct loss for binary classification
- **`metrics=['accuracy']`** — displayed during training for monitoring

In [ ]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

### Train the CNN

```python
cnn.fit(x=training_set, validation_data=test_set, epochs=25)
```

Unlike the ANN where we passed numpy arrays, here we pass `ImageDataGenerator` objects. Keras calls `.next()` on them to get batches of images from disk during each training step.

**`validation_data=test_set`** — After each epoch, Keras evaluates the model on the test set and reports `val_accuracy` and `val_loss`. This lets you monitor overfitting in real time:

- `accuracy` rising, `val_accuracy` rising together → healthy training
- `accuracy` rising, `val_accuracy` plateau or falling → overfitting, consider more augmentation or dropout

**Training time:** 25 epochs on 8,000 images requires a GPU for reasonable speed. On CPU, expect 10-30 minutes.

In [ ]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

## Part 4: Make a Single Prediction

This demonstrates the complete inference pipeline for a new image from disk.

**Steps required:**

1. **Load the image** at the same size used during training (64x64)
2. **Convert to array** — Keras works with numpy arrays, not PIL Image objects
3. **Add batch dimension** — `np.expand_dims(img, axis=0)` converts shape `(64, 64, 3)` to `(1, 64, 64, 3)`. The model expects a batch of images, even for a single prediction.
4. **Rescale** — divide by 255 (same as training preprocessing) — wait, the `ImageDataGenerator` rescaled during training but here we feed raw pixel values. The code here does NOT apply `rescale=1./255` to the single prediction image — this is a common source of errors. In production, apply the same preprocessing as training.
5. **Check class indices** — `training_set.class_indices` tells you which integer (0 or 1) corresponds to `cats` vs `dogs`, so you can correctly interpret `result[0][0] == 1`.

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
test_image = image.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = cnn.predict(test_image)
training_set.class_indices
if result[0][0] == 1:
  prediction = 'dog'
else:
  prediction = 'cat'

In [ ]:
print(prediction)